In [6]:
import re
from pydantic import BaseModel, Field
from typing import Optional
from langgraph.graph import StateGraph, END
from typing import Optional

In [7]:


from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


loader = TextLoader("loan_rules.txt")
documents = loader.load()


splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = splitter.split_documents(documents)


vectorstore = FAISS.from_documents(docs, embeddings)


vectorstore.save_local("loan_policy_index")

print("✅ RAG index created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ RAG index created


In [8]:
class LoanState(BaseModel):

SyntaxError: incomplete input (1531790405.py, line 1)

In [4]:
def query_policies(question: str) -> str:
    db = FAISS.load_local(
        "loan_policy_index",
        embeddings,
        allow_dangerous_deserialization=True
    )

    results = db.similarity_search(question, k=3)
    return "\n".join([doc.page_content for doc in results])

In [5]:
def collect_financials(state: LoanState):
    text = state.user_input.lower()

    salary = re.search(r'(\d+)\s*[kK]', text)
    credit = re.search(r'credit score\s*(\d+)', text)
    loan = re.search(r'(\d+)\s*lakh', text)
    tenure = re.search(r'(\d+)\s*years?', text)
    emi = re.search(r'emi\s*(\d+)', text)

    if salary:
        state.salary = int(salary.group(1)) * 1000

    if credit:
        state.credit_score = int(credit.group(1))

    if loan:
        state.loan_amount = int(loan.group(1)) * 100000

    if tenure:
        state.tenure_years = int(tenure.group(1))

    if emi:
        state.existing_emi = int(emi.group(1))

    return state

NameError: name 'LoanState' is not defined

In [6]:
def collect_financials(state: LoanState):
    return state

In [7]:
def collect_financials(state: LoanState):
    text = state.user_input.lower()

    salary = re.search(r'(\d+)\s?k', text)
    credit = re.search(r'credit score\s*(\d+)', text)
    loan = re.search(r'(\d+)\s?lakhs?', text)
    tenure = re.search(r'(\d+)\s?years?', text)
    emi = re.search(r'emi\s*(\d+)', text)

    if salary:
        state.salary = int(salary.group(1)) * 1000
    if credit:
        state.credit_score = int(credit.group(1))
    if loan:
        state.loan_amount = int(loan.group(1)) * 100000
    if tenure:
        state.tenure_years = int(tenure.group(1))
    if emi:
        state.existing_emi = int(emi.group(1)) * 1000

    return state

In [8]:
def check_eligibility(state: LoanState):

    rules = query_policies("loan eligibility rules")
    state.retrieved_rules = rules

    # Basic checks
    if state.salary < 25000:
        state.eligibility_status = "Not Eligible (Low Salary)"
        return state

    if state.credit_score < 700:
        state.eligibility_status = "Not Eligible (Low Credit Score)"
        return state

    # DTI Calculation
    dti = (state.existing_emi / state.salary) * 100

    if dti > 55:
        state.eligibility_status = "Not Eligible (High DTI)"
    else:
        state.eligibility_status = "Eligible"

    return state

In [9]:
def predict_approval_chance(state: LoanState):

    score = 50

    if state.credit_score > 750:
        score += 20
    elif state.credit_score > 700:
        score += 10

    if state.salary > 50000:
        score += 10

    dti = (state.existing_emi / state.salary) * 100

    if dti < 30:
        score += 10
    elif dti > 50:
        score -= 10

    state.approval_score = min(score, 100)

    return state

In [10]:
def suggest_loan_plan(state: LoanState):

    P = state.loan_amount
    r = 0.10 / 12  # 10% yearly interest
    n = state.tenure_years * 12

    emi = int((P * r * (1+r)**n) / ((1+r)**n - 1))

    state.suggested_plan = f"""
Loan Amount: ₹{P}
Tenure: {state.tenure_years} years
Estimated EMI: ₹{emi}
"""

    return state

In [11]:
def review_response(state: LoanState):

    state.final_reply = f"""
Loan Eligibility: {state.eligibility_status}

Approval Chance: {state.approval_score}%

Recommended Plan:
{state.suggested_plan}

Suggestion:
- Maintain good credit score
- Avoid high EMI burden
- Ensure stable income
"""

    return state

In [12]:
workflow = StateGraph(LoanState)

In [13]:
workflow.add_node("collect_financials", collect_financials)
workflow.add_node("check_eligibility", check_eligibility)
workflow.add_node("predict_approval_chance", predict_approval_chance)
workflow.add_node("suggest_loan_plan", suggest_loan_plan)
workflow.add_node("review_response", review_response)

In [14]:
workflow.set_entry_point("collect_financials")

In [15]:
workflow.add_edge("collect_financials", "check_eligibility")
workflow.add_edge("check_eligibility", "predict_approval_chance")
workflow.add_edge("predict_approval_chance", "suggest_loan_plan")
workflow.add_edge("suggest_loan_plan", "review_response")
workflow.add_edge("review_response", END)

In [16]:
loan_app = workflow.compile()

In [17]:
def ask_loan_ai(user_query: str) -> str:
    try:
        result = loan_app.invoke({"user_input": user_query})
        return result.get("final_reply", "No response generated.")
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:

if __name__ == "__main__":
    print("💬 Loan Eligibility Assistant (type 'exit' to quit)\n")

    while True:
        user_query = input("Enter your details: ")

        if user_query.lower() in ["exit", "quit"]:
            print("👋 Exiting...")
            break

        response = ask_loan_ai(user_query)
        print("\n🤖 Response:\n", response)
        print("-" * 50)

In [ ]:
 you have to it for yourself as nobody coming to do for what this thhe real ai chatb
where we have to learn about the bank policy and the a
chat assistant where we are using the langchain which automatically a framework of the ai tool model.